<a href="https://colab.research.google.com/github/balla-a12/quant-equity-research/blob/main/03_gov_contracts_signal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 03 — Government-Contracts Signal

This is the project's second alternative-data signal, and the first of the three that
join the congressional signal in the composite. It follows the same `BaseSignal`
contract, so the backtester from notebook `02` and the composite scorer treat it
identically to the congress signal — a $0$–$100$ score per ticker, comparable across
datasets.

### Signal design

The thesis: a company winning federal contracts, especially an accelerating flow of
them across several agencies, has a fundamental tailwind that the market may price in
slowly. The score blends four features, each a testable piece of that idea:

| Feature | Hypothesis |
|---|---|
| `award_value` | More recent award dollars signal a stronger tailwind |
| `acceleration` | Award flow rising above its prior run-rate is newly informative |
| `n_awards` | Repeated distinct wins indicate durable demand, not one lucky bid |
| `agency_breadth` | Demand across many agencies is broader than a single-buyer relationship |

Awards key on the announcement date, public when it posts, so the signal carries no
look-ahead — the same discipline the congress signal follows.


## Setup

Defaults to mock data, so it runs with no key. Set `USE_LIVE = True` for live Quiver
government-contract data.


In [1]:
!pip install -q quiverquant pandas numpy sqlalchemy plotly pyyaml requests

import subprocess, os, sys
from google.colab import userdata

GITHUB_USER = "balla-a12"
REPO = "quant-equity-research"

gh_token = userdata.get("GITHUB_TOKEN")
url = f"https://{gh_token}@github.com/{GITHUB_USER}/{REPO}.git"
r = subprocess.run(["git", "clone", url], capture_output=True, text=True)
if r.returncode == 0 or "already exists" in r.stderr:
    os.chdir(REPO)
subprocess.run(["git", "pull", "--rebase", "origin", "main"], capture_output=True, text=True)
print("Working in:", os.getcwd())

try:
    QUIVER_TOKEN = userdata.get("QUIVER_API_KEY")
except Exception:
    QUIVER_TOKEN = None
USE_LIVE = True   # flip to True for live Quiver data

Working in: /content/quant-equity-research


## Write the signal module

The signal goes into `src/quant_research/signals/`. The ingestion updates give the
mock government-contract feed a configurable history, matching how the congress feed
already works, so this signal can be backtested with the same engine later.


In [2]:
open("src/quant_research/signals/govcontracts.py", "w").write('"""Government-contract award signal.\n\nHypothesis: a company winning federal contracts — especially an accelerating flow of\nthem, across several agencies — has a fundamental tailwind the market may not have\nfully priced. The score rewards the level of recent award dollars, their acceleration\nover the prior run-rate, the number of distinct wins, and the breadth of awarding\nagencies.\n\nFeatures (each normalized across the cross-section, then weighted):\n  award_value    - total award dollars in the lookback window\n  acceleration   - recent award dollars above the prior run-rate\n  n_awards       - number of distinct awards (repeated wins)\n  agency_breadth - number of distinct awarding agencies\n\nAwards key on the announcement date, which is public when it posts, so the signal\ncarries no look-ahead.\n"""\nfrom datetime import date\nimport pandas as pd\n\nfrom .base import BaseSignal\n\nDEFAULT_WEIGHTS = {\n    "award_value": 0.40,\n    "acceleration": 0.30,\n    "n_awards": 0.20,\n    "agency_breadth": 0.10,\n}\n\n\nclass GovContractsSignal(BaseSignal):\n    name = "gov_contracts"\n    description = "Federal contract-award flow, weighted by acceleration and breadth."\n\n    def __init__(self, client, lookback_days=90, recent_days=30, weights=None):\n        self.client = client\n        self.lookback_days = lookback_days\n        self.recent_days = recent_days\n        self.weights = weights or dict(DEFAULT_WEIGHTS)\n\n    def compute(self, as_of=None, contracts=None):\n        as_of = pd.Timestamp(as_of or date.today())\n        start = as_of - pd.Timedelta(days=self.lookback_days)\n        recent_start = as_of - pd.Timedelta(days=self.recent_days)\n\n        df = self.client.gov_contracts() if contracts is None else contracts\n        win = df[(df.award_date > start) & (df.award_date <= as_of)].copy()\n        if win.empty:\n            return pd.DataFrame(columns=["score"])\n\n        g = win.groupby("ticker")\n        idx = g.size().index\n        recent = win[win.award_date > recent_start].groupby("ticker").amount.sum().reindex(idx).fillna(0)\n        prior = win[win.award_date <= recent_start].groupby("ticker").amount.sum().reindex(idx).fillna(0)\n        prior_days = max(self.lookback_days - self.recent_days, 1)\n        prior_rate = prior * (self.recent_days / prior_days)   # put prior on a recent-window footing\n\n        feat = pd.DataFrame({\n            "award_value": g.amount.sum(),\n            "acceleration": recent - prior_rate,\n            "n_awards": g.size(),\n            "agency_breadth": g.agency.nunique(),\n        })\n\n        norm = pd.DataFrame({f: self._norm(feat[f]) for f in self.weights})\n        raw = sum(self.weights[f] * norm[f] for f in self.weights)\n        feat["score"] = self._scale_0_100(raw).round(1)\n        for f in norm.columns:\n            feat[f + "_n"] = norm[f].round(3)\n        return feat.sort_values("score", ascending=False)\n\n    @staticmethod\n    def _norm(s):\n        s = s.astype(float)\n        spread = s.max() - s.min()\n        return (s - s.min()) / spread if spread > 0 else s * 0.0\n')
open("src/quant_research/ingestion/client.py", "w").write('"""A unified client for Quiver Quantitative data.\n\nThe QuiverClient exposes one method per dataset. Each method fetches raw data\n(from the live API when a token is supplied, otherwise from the mock generator)\nand returns it normalized into a consistent internal schema. The rest of the\nproject depends only on that internal schema, so nothing downstream needs to\nknow or care whether the data came from the live API or the mock source.\n"""\nimport re\nimport pandas as pd\n\nfrom . import mock_data\n\n\ndef _parse_range(text):\n    """Turn a Quiver amount range like \'$15,001 - $50,000\' into (low, high)."""\n    nums = re.findall(r"[\\d,]+", str(text))\n    vals = [int(n.replace(",", "")) for n in nums if n.replace(",", "").isdigit()]\n    if len(vals) >= 2:\n        return vals[0], vals[1]\n    if len(vals) == 1:\n        return vals[0], vals[0]\n    return 0, 0\n\n\nclass QuiverClient:\n    def __init__(self, token=None, mock=False, mock_history_days=40):\n        self.mock_history_days = mock_history_days\n        # No token means we cannot reach the live API, so we fall back to mock.\n        self.mock = mock or token is None\n        self._api = None\n        if not self.mock:\n            import quiverquant            # imported lazily; unused in mock mode\n            self._api = quiverquant.quiver(token)\n\n    # ---- Congressional trades -------------------------------------------\n    def congress_trades(self, historical=False):\n        if self.mock:\n            raw = mock_data.mock_congress_trading(\n                history_days=self.mock_history_days,\n                n=max(180, self.mock_history_days * 4))\n        else:\n            raw = self._api.congress_trading(recent=not historical)\n        return self._normalize_congress(raw)\n\n    @staticmethod\n    def _col(df, *names, default=""):\n        """First present column among `names`, else a default-filled Series.\n\n        The recent and bulk Quiver endpoints differ in their column names, so\n        every field is looked up tolerantly rather than assumed.\n        """\n        for n in names:\n            if n in df.columns:\n                return df[n]\n        return pd.Series([default] * len(df), index=df.index)\n\n    def _normalize_congress(self, df):\n        df = df.copy()\n        if "Range" in df.columns:\n            lows, highs = zip(*df["Range"].map(_parse_range)) if len(df) else ([], [])\n            amount_min, amount_max = list(lows), list(highs)\n        else:\n            amt = (self._col(df, "Amount", "Trade_Size_USD", "amount", default=0)\n                   .astype(str).str.replace(r"[$,]", "", regex=True))\n            amt = pd.to_numeric(amt, errors="coerce").fillna(0)\n            amount_min, amount_max = amt.tolist(), amt.tolist()\n        out = pd.DataFrame({\n            "ticker": self._col(df, "Ticker").astype(str).str.upper(),\n            "transaction_date": pd.to_datetime(self._col(df, "TransactionDate", "Traded"),\n                                               errors="coerce"),\n            "report_date": pd.to_datetime(self._col(df, "ReportDate", "Filed"),\n                                          errors="coerce"),\n            "representative": self._col(df, "Representative", "Name"),\n            "party": self._col(df, "Party"),\n            "chamber": self._col(df, "House", "Chamber"),\n            "transaction_type": self._col(df, "Transaction").astype(str).str.strip().str.title(),\n            "amount_min": amount_min,\n            "amount_max": amount_max,\n        })\n        return out\n\n    # ---- Insider transactions -------------------------------------------\n    def insider_trades(self):\n        raw = (mock_data.mock_insiders() if self.mock\n               else self._api.insiders())\n        return self._normalize_insiders(raw)\n\n    def _normalize_insiders(self, df):\n        df = df.copy()\n        code_map = {"P": "Purchase", "S": "Sale"}\n        out = pd.DataFrame({\n            "ticker": df["Ticker"].str.upper(),\n            "transaction_date": pd.to_datetime(df["Date"]),\n            "insider_name": df["Name"],\n            "title": df["Title"],\n            "transaction_type": df["TransactionCode"].map(code_map).fillna("Other"),\n            "shares": df["Shares"].astype(int),\n            "price_per_share": df["PricePerShare"].astype(float),\n        })\n        out["value"] = (out["shares"] * out["price_per_share"]).round(2)\n        return out\n\n    # ---- Government contracts --------------------------------------------\n    def gov_contracts(self):\n        if self.mock:\n            days = max(self.mock_history_days, 120)\n            raw = mock_data.mock_gov_contracts(history_days=days, n=max(80, days * 2))\n        else:\n            raw = self._api.gov_contracts()\n        return self._normalize_gov(raw)\n\n    def _normalize_gov(self, df):\n        df = df.copy()\n        out = pd.DataFrame({\n            "ticker": df["Ticker"].str.upper(),\n            "award_date": pd.to_datetime(df["Date"]),\n            "amount": df["Amount"].astype(float),\n            "agency": df["Agency"],\n        })\n        return out\n\n    # ---- Lobbying --------------------------------------------------------\n    def lobbying(self):\n        raw = (mock_data.mock_lobbying() if self.mock\n               else self._api.lobbying())\n        return self._normalize_lobbying(raw)\n\n    def _normalize_lobbying(self, df):\n        df = df.copy()\n        out = pd.DataFrame({\n            "ticker": df["Ticker"].str.upper(),\n            "filing_date": pd.to_datetime(df["Date"]),\n            "amount": df["Amount"].astype(float),\n            "client": df["Client"],\n            "issue": df["Issue"],\n        })\n        return out\n')
open("src/quant_research/ingestion/mock_data.py", "w").write('"""Synthetic data shaped like the Quiver Quantitative API responses.\n\nColumn names and value formats mirror the live `quiverquant` package so the same\nnormalization works on mock and real data. A few tickers carry heavy, purchase-\nskewed activity, and some buys are routed to representatives whose committee aligns\nwith the traded sector, so the signal\'s features have structure to detect. Member\nnames are real, so the live enrichment layer resolves them directly.\n"""\nimport numpy as np\nimport pandas as pd\nfrom datetime import date, timedelta\n\nfrom ..enrichment import mock as menr\n\n_UNIVERSE = ["PLTR", "LMT", "RTX", "AXON", "NOC", "CELH", "MELI", "CAVA",\n             "NVDA", "AAPL", "MSFT", "GE", "BA", "CAT", "JPM"]\n_HOT = ["PLTR", "LMT", "AXON"]\n_RANGES = ["$1,001 - $15,000", "$15,001 - $50,000", "$50,001 - $100,000",\n           "$100,001 - $250,000", "$250,001 - $500,000", "$500,001 - $1,000,000"]\n_REPS = list(menr.REP_COMMITTEE.keys())\n_TITLES = ["CEO", "CFO", "Director", "President", "EVP", "VP Finance", "COO"]\n_AGENCIES = ["Department of Defense", "Department of Energy", "NASA",\n             "Department of Homeland Security", "Department of Health"]\n_ISSUES = ["Defense", "Healthcare", "Taxation", "Energy", "Technology", "Trade"]\n\n\ndef _recent(rng, max_days_ago, min_days_ago=0):\n    return date.today() - timedelta(days=int(rng.integers(min_days_ago, max_days_ago)))\n\n\ndef mock_congress_trading(seed=42, n=180, history_days=40):\n    rng = np.random.default_rng(seed)\n    rows = []\n    for _ in range(n):\n        if rng.random() < 0.45:\n            ticker = rng.choice(_HOT)\n            txn = rng.choice(["Purchase", "Sale"], p=[0.85, 0.15])\n        else:\n            ticker = rng.choice(_UNIVERSE)\n            txn = rng.choice(["Purchase", "Sale"], p=[0.55, 0.45])\n\n        sector = menr.TICKER_SECTOR.get(ticker)\n        aligned = menr.SECTOR_REPS.get(sector, [])\n        rep = rng.choice(aligned) if (aligned and rng.random() < 0.5) else rng.choice(_REPS)\n\n        report = _recent(rng, history_days)\n        transaction = report - timedelta(days=int(rng.integers(10, 45)))\n        rows.append({\n            "Representative": rep,\n            "Party": "D" if _REPS.index(rep) % 2 == 0 else "R",\n            "House": rng.choice(["Representatives", "Senate"], p=[0.75, 0.25]),\n            "Ticker": ticker, "Transaction": txn, "Range": rng.choice(_RANGES),\n            "TransactionDate": transaction, "ReportDate": report,\n        })\n    return pd.DataFrame(rows)\n\n\ndef mock_insiders(seed=43, n=120):\n    rng = np.random.default_rng(seed)\n    rows = []\n    for _ in range(n):\n        ticker = rng.choice(_HOT) if rng.random() < 0.4 else rng.choice(_UNIVERSE)\n        rows.append({\n            "Ticker": ticker, "Name": f"Insider {rng.integers(1000, 9999)}",\n            "Title": rng.choice(_TITLES), "Date": _recent(rng, 90),\n            "TransactionCode": rng.choice(["P", "S"], p=[0.6, 0.4]),\n            "Shares": int(rng.integers(500, 50000)),\n            "PricePerShare": round(float(rng.uniform(20, 400)), 2),\n        })\n    return pd.DataFrame(rows)\n\n\ndef mock_gov_contracts(seed=44, n=80, history_days=120):\n    rng = np.random.default_rng(seed)\n    rows = []\n    for _ in range(n):\n        ticker = rng.choice(_HOT) if rng.random() < 0.5 else rng.choice(_UNIVERSE)\n        rows.append({\n            "Ticker": ticker, "Date": _recent(rng, history_days),\n            "Amount": int(rng.integers(50_000, 500_000_000)),\n            "Agency": rng.choice(_AGENCIES), "Description": "Procurement contract award",\n        })\n    return pd.DataFrame(rows)\n\n\ndef mock_lobbying(seed=45, n=140):\n    rng = np.random.default_rng(seed)\n    rows = []\n    for _ in range(n):\n        ticker = rng.choice(_UNIVERSE)\n        rows.append({\n            "Ticker": ticker, "Date": _recent(rng, 360),\n            "Amount": int(rng.integers(10_000, 5_000_000)),\n            "Client": f"{ticker} Inc.", "Issue": rng.choice(_ISSUES),\n        })\n    return pd.DataFrame(rows)\n')
print("wrote gov-contracts signal and ingestion updates")

wrote gov-contracts signal and ingestion updates


In [3]:
!pip install -q -e .

import os, sys, importlib
src_path = os.path.abspath("src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)
import quant_research; importlib.reload(quant_research)
print("Package ready:", quant_research.__version__)

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for quant-equity-research (pyproject.toml) ... done
Package ready: 0.1.0


## Construct the signal

We compute the signal as of today over a $90$-day lookback, treating the most recent
$30$ days as the window whose flow is compared against the prior run-rate. `compute`
returns a DataFrame indexed by ticker, sorted by score, with the features behind it.


In [4]:
from quant_research.ingestion.client import QuiverClient
from quant_research.signals.govcontracts import GovContractsSignal
import pandas as pd

client = QuiverClient(token=QUIVER_TOKEN) if USE_LIVE else QuiverClient(mock=True)
signal = GovContractsSignal(client, lookback_days=90, recent_days=30)

scores = signal.compute()
print(f"{len(scores)} tickers scored | data: {'live' if USE_LIVE else 'mock'}")

disp = scores.head(12).copy()
disp["award_$M"] = (disp["award_value"] / 1e6).round(1)
disp["accel_$M"] = (disp["acceleration"] / 1e6).round(1)
disp[["score", "award_$M", "accel_$M", "n_awards", "agency_breadth"]]

253 tickers scored | data: live


,score,award_$M,accel_$M,n_awards,agency_breadth
ticker,,,,,
TXT,100.0,218.3,204.1,9,6
F,83.5,229.2,-101.9,4387,1
GD,77.6,191.3,66.9,26,10
BAH,72.9,317.9,-138.6,40,10
MRK,60.4,363.6,-179.0,8,2
TPC,57.4,225.3,10.1,5,1
MDLN,55.6,179.0,36.0,99,3
DNOW,43.7,3.4,-0.1,3797,3
PLTR,43.4,214.4,-100.8,9,5


### Reading the output

`score` is the $0$–$100$ conviction reading. The `*_n` columns are the features
normalized across the cross-section — the actual inputs to the weighted score, so each
score is fully explainable.


In [5]:
comp_cols = [c for c in scores.columns if c.endswith("_n")]
scores[["score"] + comp_cols].head(8)

,score,award_value_n,acceleration_n,n_awards_n,agency_breadth_n
ticker,,,,,
TXT,100.0,0.600,1.000,0.002,0.455
F,83.5,0.630,0.201,1.000,0.000
GD,77.6,0.526,0.642,0.006,0.818
BAH,72.9,0.874,0.105,0.009,0.818
MRK,60.4,1.000,0.000,0.002,0.091
TPC,57.4,0.620,0.494,0.001,0.000
MDLN,55.6,0.492,0.561,0.022,0.182
DNOW,43.7,0.009,0.467,0.865,0.182


### Top names, shaded by acceleration

The chart shades each candidate by acceleration, so you can see whether a high score
comes from a large but steady award base or from a genuine pickup in recent flow.


In [6]:
import plotly.express as px

top = scores.head(15).reset_index()
fig = px.bar(top, x="score", y="ticker", orientation="h",
             color="acceleration_n", color_continuous_scale="Oranges",
             labels={"acceleration_n": "acceleration"},
             title="Gov-contracts signal — top 15 (shading = acceleration)")
fig.update_layout(yaxis=dict(autorange="reversed"), height=460)
fig.show()

## Reading it honestly

A few caveats specific to this signal:

- **Awards are lumpy.** A single large contract can dominate a ticker's score for a
  quarter, so acceleration and agency breadth matter as much as raw dollars — a broad,
  rising flow is more durable than one headline award.
- **A contract is not booked revenue.** Announced award value is a ceiling that pays
  out over time and can be reduced or cancelled, so treat the dollar figures as a
  signal of momentum rather than a fundamental estimate.
- **No enrichment needed here.** Unlike the congress signal, this one needs no net
  worth or committee data — the award flow speaks for the company directly.

The same event-study engine from notebook `02` validates this signal without changes,
since it depends only on the `BaseSignal` contract. We'll run that once the full signal
set is in place.


## Commit

In [7]:
!git config --global user.email "aballa1234@gmail.com"
!git config --global user.name "Alex Balla"

!git add -A
!git commit -m "Add government-contracts signal"
!git push

[main a688eae] Add government-contracts signal
 3 files changed, 83 insertions(+), 4 deletions(-)
 create mode 100644 src/quant_research/signals/govcontracts.py
Enumerating objects: 16, done.
Counting objects: 100% (16/16), done.
Delta compression using up to 2 threads
Compressing objects: 100% (8/8), done.
Writing objects: 100% (9/9), 2.04 KiB | 2.04 MiB/s, done.
Total 9 (delta 5), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (5/5), completed with 5 local objects.
To https://github.com/balla-a12/quant-equity-research.git
   32d2d31..a688eae  main -> main


## Recap and next

The second signal is built on the shared contract: a transparent government-contract
score driven by award level, acceleration, repeat wins, and agency breadth, with the
same disclosure-date discipline as the congress signal.

Next come the lobbying and off-exchange signals on the same pattern, then the composite
scorer that blends all four into one ranking, and a combined backtest of the composite
through the `02` engine. The dashboard notebook then surfaces today's candidates with
the historical edge behind each.
